In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()
gemini_api_key=os.getenv("GEMINI_API_KEY")
groq_api_key=os.getenv("GROQ_API_KEY")
hf_api_key=os.getenv("HF_API_KEY")

e:\ananconda3\envs\generativeai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm=ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

In [41]:
summarize_middleware=SummarizationMiddleware(
    model=llm,
    trigger=("messages",50),
    keep=("messages",20)
)

In [42]:
agent=create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[summarize_middleware]
)

In [43]:
config={"configurable":{"thread_id":"test-1"}}

In [44]:
test_data = [
    "What is  2+2",
    "What is  5+3",
    "What is  10-4",
    "What is  6*7",
    "What is  8/2",
    "What is  9+11",
    "What is  15-5",
    "What is  3*4",
    "What is  20/5",
    "What is  7+6"
]

In [45]:
for t in test_data:
    response=agent.invoke({"messages":[HumanMessage(content=t)]},config=config)
    print(f"Messages {response}")
    print(f"Messages {len(response['messages'])}")
    

Messages {'messages': [HumanMessage(content='What is  2+2', additional_kwargs={}, response_metadata={}, id='e453a6a1-e925-4353-8156-8af3d218e4c9'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks a simple math question: "What is 2+2". We respond with answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 78, 'total_tokens': 122, 'completion_time': 0.092705091, 'completion_tokens_details': {'reasoning_tokens': 25}, 'prompt_time': 0.002898476, 'prompt_tokens_details': None, 'queue_time': 0.050765064, 'total_time': 0.095603567}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a618bed98', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d8727-b02c-7411-8dc7-56a41d7b8025-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 44, 'total_tokens': 122, 'output_token_details': {'rea

In [47]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def your_read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def your_send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=llm,
    tools=[your_read_email_tool, your_send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "your_send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "your_read_email_tool": False,
            }
        ),
    ],
)

In [48]:
config={"configurable":{"thread_id":"test_approve"}}

In [49]:
result=agent.invoke({"messages":[HumanMessage(content="""send email to lakshayvarshney62@gmail.com with subject "hello" and body How are you""")]},config=config)

In [50]:
result

{'messages': [HumanMessage(content='send email to lakshayvarshney62@gmail.com with subject "hello" and body How are you', additional_kwargs={}, response_metadata={}, id='858474db-dab5-4131-b105-cd62c7b1225b'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using tool. Use send_email.', 'tool_calls': [{'id': 'fc_0f7866f9-eeb0-4173-ac72-d44db8edb5e2', 'function': {'arguments': '{"body":"How are you","recipient":"lakshayvarshney62@gmail.com","subject":"hello"}', 'name': 'your_send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 179, 'total_tokens': 245, 'completion_time': 0.142514144, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.007188793, 'prompt_tokens_details': None, 'queue_time': 0.051052191, 'total_time': 0.149702937}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e10890e4b9', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 

In [58]:

from langgraph.types import Command

config = {"configurable": {"thread_id": "1"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Send email"}]},
    config=config
)

if isinstance(result, dict) and result.get("__interrupt__"):
    
    interrupts = result["__interrupt__"]

    decisions = []
    for interrupt in interrupts:
        decisions.append({
            "type": "approve",
            "id": interrupt["id"]
        })

    result = agent.invoke(
        Command(
            resume={
                "decisions": decisions
            }
        ),
        config=config
    )

print(result)

{'messages': [HumanMessage(content='Send email', additional_kwargs={}, response_metadata={}, id='3ed4053f-39aa-4202-b59b-6206b90fd973'), AIMessage(content='Sure, I can help with that. Could you please provide the following details?\n\n1. **Recipient’s email address**  \n2. **Subject line**  \n3. **Message body** (the content you’d like to include)\n\nLet me know, and I’ll send the email for you!', additional_kwargs={'reasoning_content': 'We need to send an email, but we need details: recipient, subject, body. The user just said "Send email". We need to ask clarification.'}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 160, 'total_tokens': 262, 'completion_time': 0.222625517, 'completion_tokens_details': {'reasoning_tokens': 33}, 'prompt_time': 0.019834873, 'prompt_tokens_details': None, 'queue_time': 0.064129935, 'total_time': 0.24246039}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_626f3fc5e0', 'service_tier': 'on_demand', 'finish_re

In [56]:
result.get("__interrupt__")

[Interrupt(value={'action_requests': [{'name': 'your_send_email_tool', 'args': {'body': 'How are you', 'recipient': 'lakshayvarshney62@gmail.com', 'subject': 'hello'}, 'description': "Tool execution requires approval\n\nTool: your_send_email_tool\nArgs: {'body': 'How are you', 'recipient': 'lakshayvarshney62@gmail.com', 'subject': 'hello'}"}], 'review_configs': [{'action_name': 'your_send_email_tool', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='54ea80a7c67295fdafa19b34ef19f49a')]